# Sprint 4 Mejora A+B+C: Reentrenamiento completo

**Mejoras aplicadas:**

**A — Augmentación agresiva:**
- RandomHorizontalFlip (p=0.3): baja probabilidad respetando anatomía.
- RandomAffine (±10°, translate 7%, scale ±8%): variaciones geométricas.
- GaussianBlur: simula variaciones de nitidez del equipo de RX.

**B — Label Smoothing (ε=0.1):**
- Targets suaves en vez de one-hot duro.
- Reduce overconfidence, mejora calibración.

**C — Más datos (12 tarballs completos, ~80-90k imágenes):**
- Cardiomegaly: 260 → ~2,700+ pacientes únicos.
- Dataset train: ~3,900 → ~25,000+ imágenes.

**Estrategia:** Entrenar desde cero (Fase 1 + Fase 2) con todos los cambios.

**Runtime: T4 GPU. Tiempo estimado: ~2-3 horas.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠ CPU'}")

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_S4 = '/content/processed_s4v2'
CLASS_NAMES  = {0:'No Finding', 1:'Cardiomegaly', 2:'Effusion', 3:'Infiltration'}


In [ ]:
!pip install -q torchxrayvision scipy h5py
print('deps OK')


## 2. Copiar HDF5 ampliado a disco local

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    size_gb = os.path.getsize(f'{NIH_DIR}/nih_images.h5') / (1024**3)
    print(f'Copiando HDF5 ampliado ({size_gb:.1f} GB)...')
    t0 = time.time()
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)
    print(f'✓ Copiado en {(time.time()-t0)/60:.1f} min')

with h5py.File(H5_LOCAL, 'r') as hf:
    n = len(hf['images'])
print(f'✓ HDF5 local: {n:,} imágenes')
print(f'  (antes de mejora C: 34,999 | ahora: {n:,})')


## 3. Instalar mejoras A y B

In [ ]:
from pathlib import Path

# Mejora A: transforms con augmentación
content = r'''"""
src/datasets/transforms_v2.py
Transforms con augmentación agresiva para Sprint 4 (Mejora A).

Diferencias respecto a transforms.py:
  - RandomHorizontalFlip(p=0.3): con baja probabilidad para respetar
    la anatomía del tórax (corazón izquierdo, aorta).
  - RandomAffine: traslaciones y escalados pequeños.
  - GaussianBlur: simula variaciones de nitidez del equipo.
  - ColorJitter: simula variaciones de exposición del RX.

Referencia: Rajpurkar et al. (2017) CheXNet usa flip horizontal.
Nota metodológica: flip se usa con p=0.3 y se documenta en la tesis.
"""
import torchxrayvision as xrv
import torch
from torchvision import transforms
import numpy as np


class XRVToTensor:
    """Convierte array (H,W) uint8 a tensor (1,H,W) float normalizado xrv."""
    def __call__(self, img):
        if isinstance(img, np.ndarray):
            img = img.astype(np.float32)
            # Normalizar a [-1024, 1024] como espera torchxrayvision
            img = (img / 255.0) * 2048.0 - 1024.0
            return torch.from_numpy(img).unsqueeze(0)
        return img


def build_train_transform_v2(image_size: int = 224):
    """Transform de entrenamiento con augmentación agresiva."""
    return transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(image_size),
        XRVToTensor(),
        # Augmentaciones espaciales (p bajas para respetar anatomía)
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomAffine(
            degrees=10,           # ±10° de rotación
            translate=(0.07, 0.07),  # traslación máxima 7%
            scale=(0.92, 1.08),   # escala ±8%
        ),
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5)),
    ])


def build_eval_transform_v2(image_size: int = 224):
    """Transform de evaluación (sin augmentación)."""
    return transforms.Compose([
        xrv.datasets.XRayCenterCrop(),
        xrv.datasets.XRayResizer(image_size),
        XRVToTensor(),
    ])
'''
p = Path(f'{CODE_DIR}/src/datasets/transforms_v2.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


In [ ]:
# Mejora B: label smoothing
content = r'''"""
src/training/label_smoothing.py
Label Smoothing Cross-Entropy para Sprint 4 (Mejora B).

En vez de targets duros [0,0,1,0], usa targets suaves:
  [ε/(C-1), ε/(C-1), 1-ε, ε/(C-1)]  con ε=0.1, C=4

Beneficios:
  1. Reduce overconfidence del modelo.
  2. Mejora calibración de probabilidades.
  3. Actúa como regularizador adicional.

Referencia: Szegedy et al. (2016). Rethinking the Inception Architecture.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing: float = 0.1,
                 weight: torch.Tensor | None = None):
        super().__init__()
        assert 0.0 < smoothing < 1.0
        self.smoothing = smoothing
        self.weight    = weight

    def forward(self, pred: torch.Tensor,
                target: torch.Tensor) -> torch.Tensor:
        n_classes = pred.size(1)
        log_prob  = F.log_softmax(pred, dim=1)

        # Targets suaves
        smooth_val = self.smoothing / (n_classes - 1)
        with torch.no_grad():
            smooth_target = torch.full_like(log_prob, smooth_val)
            smooth_target.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)

        # Loss por muestra
        loss = -(smooth_target * log_prob).sum(dim=1)  # (B,)

        # Aplicar class weights si se proporcionaron
        if self.weight is not None:
            w = self.weight.to(pred.device)[target]
            loss = (loss * w).mean()
        else:
            loss = loss.mean()

        return loss
'''
p = Path(f'{CODE_DIR}/src/training/label_smoothing.py')
p.write_text(content, encoding='utf-8')
print(f'OK -> {p}')


## 4. Generar splits Sprint 4 v2 (con datos ampliados)

In [ ]:
import subprocess

result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/prepare_splits_s4.py'],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr[-3000:])


In [ ]:
import pandas as pd

# Copiar a disco local
os.makedirs(PROCESSED_S4, exist_ok=True)
for csv in ['train.csv', 'val.csv', 'test.csv']:
    # Los splits se generan en data/processed_s4/
    shutil.copy(f'{PROJECT_ROOT}/data/processed_s4/{csv}',
                f'{PROCESSED_S4}/{csv}')
    df = pd.read_csv(f'{PROCESSED_S4}/{csv}')
    dist = {CLASS_NAMES[l]: int((df['label']==l).sum()) for l in range(4)}
    print(f'{csv}: {len(df)} imgs | {dist}')

df_s2 = pd.read_csv(f'{PROJECT_ROOT}/data/processed_s2/train.csv')
df_s4 = pd.read_csv(f'{PROCESSED_S4}/train.csv')
print(f'\nComparativa train:')
print(f'  Sprint 2:  {len(df_s2):>6} imgs')
print(f'  Sprint 4v2: {len(df_s4):>6} imgs ({len(df_s4)/len(df_s2):.1f}x)')


In [ ]:
import h5py, os

H5_DRIVE = '/content/drive/MyDrive/Tesis_CXR/data/raw/nih/nih_images.h5'
H5_LOCAL = '/content/nih_images.h5'

# Ver qué tiene cada uno
with h5py.File(H5_DRIVE, 'r') as hf:
    n_drive = len(hf['images'])
with h5py.File(H5_LOCAL, 'r') as hf:
    n_local = len(hf['images'])

print(f'HDF5 en Drive:  {n_drive:,} imágenes')
print(f'HDF5 en Local:  {n_local:,} imágenes')
print()

# Deducir qué tarballs están completos
if n_drive < 40000:
    print('Solo tarballs 001-005 en Drive.')
elif n_drive < 55000:
    print('Tarballs 001-006 en Drive.')
elif n_drive < 65000:
    print('Tarballs 001-007 en Drive.')
elif n_drive < 75000:
    print('Tarballs 001-008 en Drive.')
elif n_drive < 80000:
    print('Tarballs 001-009 en Drive.')
elif n_drive < 90000:
    print('Tarballs 001-010 en Drive.')
elif n_drive < 100000:
    print('Tarballs 001-011 en Drive.')
else:
    print('Todos los tarballs (001-012) en Drive.')

print()
# Cuántos faltan
faltan = 102120 - n_drive
print(f'Imágenes faltantes: {faltan:,}')
print(f'Tarballs pendientes aproximados: {faltan // 10000 + 1}')

## 5. Smoke test: augmentación y label smoothing

In [ ]:
# CELDA FIX: reemplazar transforms_v2.py con versión corregida
from pathlib import Path
import numpy as np, torch
from torchvision import transforms

FIX = '''
import numpy as np
import torch
from torchvision import transforms


def build_train_transform_v2(image_size: int = 224):
    """Transform con augmentación. Evita XRayCenterCrop (no soporta 2D en versiones nuevas)."""
    return transforms.Compose([
        # (H,W) uint8 → (1,H,W) float tensor normalizado a [-1024, 1024]
        transforms.Lambda(lambda x: torch.from_numpy(
            x.astype(np.float32)).unsqueeze(0) / 255.0 * 2048.0 - 1024.0),
        transforms.CenterCrop(min(image_size * 256 // 224, 256)),
        transforms.Resize(image_size, antialias=True),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomAffine(degrees=10, translate=(0.07, 0.07), scale=(0.92, 1.08)),
        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5)),
    ])


def build_eval_transform_v2(image_size: int = 224):
    """Transform de evaluación sin augmentación."""
    return transforms.Compose([
        transforms.Lambda(lambda x: torch.from_numpy(
            x.astype(np.float32)).unsqueeze(0) / 255.0 * 2048.0 - 1024.0),
        transforms.CenterCrop(min(image_size * 256 // 224, 256)),
        transforms.Resize(image_size, antialias=True),
    ])
'''

p = Path(f'{CODE_DIR}/src/datasets/transforms_v2.py')
p.write_text(FIX, encoding='utf-8')
print(f'✓ transforms_v2.py corregido')

In [ ]:
# Recargar módulos
for mod in list(__import__('sys').modules.keys()):
    if mod.startswith('src.'): del __import__('sys').modules[mod]

from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms_v2 import build_train_transform_v2, build_eval_transform_v2

train_tfm = build_train_transform_v2(224)
eval_tfm  = build_eval_transform_v2(224)

train_ds = NIHDatasetHDF5(f'{PROCESSED_S4}/train.csv', H5_LOCAL, train_tfm)
val_ds   = NIHDatasetHDF5(f'{PROCESSED_S4}/val.csv',   H5_LOCAL, eval_tfm)

# Test
sample = train_ds[0]
print(f'Shape: {sample["image"].shape}')
print(f'Range: [{sample["image"].min():.0f}, {sample["image"].max():.0f}]')
print(f'Label: {sample["label"].item()} | {CLASS_NAMES[sample["label"].item()]}')
print('✓ Transforms OK')

In [ ]:
# Limpiar cache
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.datasets.nih_hdf5 import NIHDatasetHDF5
from src.datasets.transforms_v2 import build_train_transform_v2, build_eval_transform_v2
from src.training.label_smoothing import LabelSmoothingCrossEntropy
import torch, matplotlib.pyplot as plt

train_tfm = build_train_transform_v2(224)
eval_tfm  = build_eval_transform_v2(224)

train_ds = NIHDatasetHDF5(f'{PROCESSED_S4}/train.csv', H5_LOCAL, train_tfm)
val_ds   = NIHDatasetHDF5(f'{PROCESSED_S4}/val.csv',   H5_LOCAL, eval_tfm)
test_ds  = NIHDatasetHDF5(f'{PROCESSED_S4}/test.csv',  H5_LOCAL, eval_tfm)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Distribución train: {train_ds.class_distribution()}')

# Mostrar 4 versiones augmentadas de la misma imagen
sample_idx = 0
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
base = val_ds[sample_idx]['image'].squeeze().numpy()
axes[0].imshow(base, cmap='gray'); axes[0].set_title('Original (sin aug)')
for i in range(1, 5):
    aug = train_ds[sample_idx]['image'].squeeze().numpy()
    axes[i].imshow(aug, cmap='gray')
    axes[i].set_title(f'Aug #{i}')
for ax in axes: ax.axis('off')
plt.suptitle(f'Augmentación — {train_ds.df.iloc[sample_idx]["Image Index"]}')
plt.tight_layout(); plt.show()

# Test label smoothing
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
dummy_pred   = torch.randn(4, 4)
dummy_target = torch.tensor([0, 1, 2, 3])
loss = criterion(dummy_pred, dummy_target)
print(f'\n✓ Label Smoothing OK: loss={loss.item():.4f}')


## 6. Entrenar CNN-ViT con A+B+C

Tiempo estimado: **~2-3 horas en T4**.


In [ ]:
import copy, numpy as np
import torch.nn as nn
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.models.cnn_vit import build_cnn_vit
from src.training.label_smoothing import LabelSmoothingCrossEntropy
from src.evaluation.metrics_s2 import compute_metrics_multiclass, logits_to_probs_multiclass
from src.training.utils import load_config, get_device, set_seed, save_checkpoint

cfg    = load_config(f'{CODE_DIR}/configs/sprint4.yaml')
device = get_device()
set_seed(42)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

# ── Class weights ──────────────────────────────────────────────
def get_weights(loader, n_cls, dev):
    counts = torch.zeros(n_cls)
    for batch in loader:
        for l in batch['label']: counts[l.item()] += 1
    total   = counts.sum()
    weights = total / (n_cls * counts)
    return (weights / weights.sum() * n_cls).to(dev)

print('Calculando class weights...')
weights = get_weights(train_loader, 4, device)
print(f'Class weights: {[f"{w:.2f}" for w in weights.tolist()]}')

# ── Criterion: Label Smoothing + class weights ─────────────────
criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=weights)
print('✓ Label Smoothing ε=0.1 + class weights activados')


In [ ]:
# ── Función de una epoch ──────────────────────────────────────

def run_epoch(model, loader, optimizer, criterion, device,
              use_amp, scaler, train=True):
    model.train() if train else model.eval()
    losses, all_probs, all_labels = [], [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)
            if train:
                optimizer.zero_grad(set_to_none=True)
            if use_amp and scaler and train:
                with autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(imgs); loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            else:
                logits = model(imgs); loss = criterion(logits, labels)
                if train: loss.backward(); optimizer.step()
            losses.append(loss.item())
            probs = logits_to_probs_multiclass(logits.detach().float())
            all_probs.append(probs.cpu()); all_labels.append(labels.cpu())
    m = compute_metrics_multiclass(
        torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()
    )
    return {'loss': float(np.mean(losses)), 'metrics': m}


def run_phase(model, train_loader, val_loader, optimizer, scheduler,
              epochs, patience, device, criterion, use_amp, scaler,
              ckpt_path, phase_name, start_auc=-float('inf')):
    best_auc, best_state, no_improve, history = start_auc, None, 0, []
    print(f'\n{"="*65}')
    print(f'FASE {phase_name} | {epochs} epochs | patience={patience}')
    print(f'Entrenables: {model.trainable_params():,}')
    print('='*65)
    for epoch in range(1, epochs+1):
        lr = optimizer.param_groups[0]['lr']
        tr = run_epoch(model, train_loader, optimizer, criterion, device,
                       use_amp, scaler, train=True)
        va = run_epoch(model, val_loader, None, criterion, device,
                       use_amp, scaler, train=False)
        if scheduler: scheduler.step()
        print(f'[{phase_name}] Epoch {epoch:2d}/{epochs} | LR={lr:.1e} | '
              f'train AUC={tr["metrics"]["auc_macro"]:.4f} | '
              f'val AUC={va["metrics"]["auc_macro"]:.4f} '
              f'acc={va["metrics"]["accuracy"]:.4f}')
        for i, auc in enumerate(va['metrics']['auc_per_class']):
            print(f'  val AUC clase {i} ({CLASS_NAMES[i]}): {auc:.4f}')
        history.append({'phase':phase_name,'epoch':epoch,'lr':lr,
                        'train_loss':tr['loss'],'train_metrics':tr['metrics'],
                        'val_loss':va['loss'],'val_metrics':va['metrics']})
        score = va['metrics']['auc_macro']
        if score > best_auc:
            best_auc = score
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save({'phase':phase_name,'epoch':epoch,
                        'model_state_dict':best_state,
                        'best_val_auc_macro':best_auc}, ckpt_path)
            print(f'  → NUEVO MEJOR: {best_auc:.4f} ✓')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping.'); break
    print(f'Fase {phase_name} terminada. Mejor AUC_macro={best_auc:.4f}')
    return best_state, history, best_auc


In [ ]:
# ── Modelo y entrenamiento ────────────────────────────────────

CKPT_DIR = f'{PROJECT_ROOT}/experiments_s4/checkpoints'
os.makedirs(f'{PROJECT_ROOT}/experiments_s4/logs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/experiments_s4/figures', exist_ok=True)

model  = build_cnn_vit(cfg).to(device)
use_amp = True
scaler  = GradScaler(device='cuda')
all_history = []

# ── FASE 1: CNN congelada ─────────────────────────────────────
model.set_phase(1)
opt1 = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=1e-4
)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=15, eta_min=1e-6)

bs1, h1, auc_p1 = run_phase(
    model, train_loader, val_loader, opt1, sched1,
    epochs=15, patience=6, device=device,
    criterion=criterion, use_amp=use_amp, scaler=scaler,
    ckpt_path=f'{CKPT_DIR}/sprint4v2_phase1_best.pt',
    phase_name='P1',
)
all_history.extend(h1)
if bs1: model.load_state_dict(bs1)

# ── FASE 2: full FT con LR diferenciado ──────────────────────
model.set_phase(2)
param_groups = model.get_param_groups(lr_cnn=2e-6, lr_vit=5e-5)
opt2 = torch.optim.Adam(param_groups, weight_decay=1e-4)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=40, eta_min=1e-7)

bs2, h2, auc_p2 = run_phase(
    model, train_loader, val_loader, opt2, sched2,
    epochs=40, patience=15, device=device,
    criterion=criterion, use_amp=use_amp, scaler=scaler,
    ckpt_path=f'{CKPT_DIR}/sprint4v2_phase2_best.pt',
    phase_name='P2',
    start_auc=auc_p1,
)
all_history.extend(h2)

import json
result = {
    'history': all_history,
    'best_val_auc_phase1': float(auc_p1),
    'best_val_auc_phase2': float(auc_p2),
    'improvements': ['augmentation_v2', 'label_smoothing_0.1',
                     'tarballs_006_012'],
}
hp = f'{PROJECT_ROOT}/experiments_s4/history_s4v2.json'
with open(hp, 'w') as f: json.dump(result, f, indent=2)
print(f'\n✓ History guardado: {hp}')
print(f'RESUMEN: P1={auc_p1:.4f} | P2={auc_p2:.4f}')


## 7. Comparativa Sprint 2 vs S4 original vs S4 A+B+C

In [ ]:
import json, matplotlib.pyplot as plt

with open(f'{PROJECT_ROOT}/experiments_s2/history_s2.json') as f:
    d2 = json.load(f)
with open(f'{PROJECT_ROOT}/experiments_s4/history_s4.json') as f:
    d4 = json.load(f)
with open(f'{PROJECT_ROOT}/experiments_s4/history_s4v2.json') as f:
    d4v2 = json.load(f)

def best_metrics(history, phase='P2'):
    h = [x for x in history if x.get('phase','') == phase]
    if not h: h = history
    return max(h, key=lambda x: x['val_metrics']['auc_macro'])['val_metrics']

m2   = best_metrics(d2['history'], phase='')
m4   = best_metrics(d4['history'], phase='P2')
m4v2 = best_metrics(d4v2['history'], phase='P2')

print('COMPARATIVA FINAL (val set):')
print('='*72)
print(f"{'Métrica':<18} {'Sprint 2':>10} {'S4 orig':>10} {'S4 A+B+C':>10} {'Delta':>8}")
print('-'*72)
for key, label in [('auc_macro','AUC macro'),('accuracy','Accuracy'),
                    ('sensitivity_macro','Sens macro'),('specificity_macro','Spec macro')]:
    v2, v4, v4v = m2.get(key,0), m4.get(key,0), m4v2.get(key,0)
    delta = v4v - v2
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f"{label:<18} {v2:>10.4f} {v4:>10.4f} {v4v:>10.4f} {delta:>+8.4f} {arrow}")

print()
print(f"{'Clase':<15} {'S2':>6} {'S4':>6} {'S4v2':>8} {'Delta':>8}")
print('-'*50)
for i in range(4):
    a2  = m2['auc_per_class'][i]
    a4  = m4['auc_per_class'][i]
    a4v = m4v2['auc_per_class'][i]
    delta = a4v - a2
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f"{CLASS_NAMES[i]:<15} {a2:>6.4f} {a4:>6.4f} {a4v:>8.4f} {delta:>+8.4f} {arrow}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3','#F44336','#4CAF50','#FF9800']

ax = axes[0]
def plot_auc(hist, label, color, style='-'):
    h = [x for x in hist if x.get('phase','') in ('P2','')]
    if not h: h = hist
    ep  = [x['epoch'] for x in h]
    auc = [x['val_metrics']['auc_macro'] for x in h]
    ax.plot(ep, auc, style, ms=3, label=f'{label} ({max(auc):.4f})', color=color)

plot_auc(d2['history'],   'Sprint 2 CNN',   'coral')
plot_auc(d4['history'],   'S4 orig CNN-ViT','steelblue', '--')
plot_auc(d4v2['history'], 'S4 A+B+C',       'steelblue')
ax.axhline(0.5,color='gray',linestyle=':',alpha=0.4)
ax.set_title('Val AUC Macro: S2 vs S4 vs S4 A+B+C')
ax.set_xlabel('Epoch'); ax.set_ylabel('AUC macro')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0, 1.05)

ax2 = axes[1]
p2v2_hist = [x for x in d4v2['history'] if x.get('phase') == 'P2']
for i in range(4):
    aucs = [x['val_metrics']['auc_per_class'][i] for x in p2v2_hist]
    ax2.plot([x['epoch'] for x in p2v2_hist], aucs, 'o-', ms=3,
             label=CLASS_NAMES[i], color=colors[i])
ax2.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
ax2.set_title('S4 A+B+C: Val AUC por clase (Fase 2)')
ax2.set_xlabel('Epoch'); ax2.legend(fontsize=8)
ax2.grid(alpha=0.3); ax2.set_ylim(0, 1.05)

best_auc_v2 = max(x['val_metrics']['auc_macro'] for x in p2v2_hist)
plt.suptitle(f'S2={m2["auc_macro"]:.4f} → S4={m4["auc_macro"]:.4f} → S4v2={best_auc_v2:.4f}')
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/experiments_s4/figures/curves_s4_ABC.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada.')
